<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>

# Demo - Multi-Agent Orchestration with the OpenAI Agents SDK
## Overview
In this demo, you will build a multi-agent system using tools already registered in Unity Catalog. The Data Analyst agent uses `classify_price_tier` and the Business Analyst agent uses `neighborhood_summary`, both accessed via Managed MCP servers. A supervisor agent routes user requests to the appropriate worker using handoffs. You will observe the full orchestration flow through MLflow distributed tracing.

## Learning Objectives

By the end of this demo, you will be able to:

1. **Connect to Unity Catalog functions via Managed MCP servers** using both `DatabricksMCPClient` (for synchronous discovery) and `McpServer` (for async agent integration)
2. **Build specialized worker agents** backed by MCP tools, each scoped to a single UC function
3. **Implement a supervisor agent** that delegates user requests to the appropriate worker using the OpenAI Agents SDK `handoff` mechanism
4. **Validate multi-agent routing** by testing queries across both worker agents and confirming correct handoff decisions
5. **Enable MLflow autologging** to capture distributed execution traces across supervisor and worker agent spans
6. **Construct a unified trace** by wrapping multi-agent flows with `@mlflow.trace` to consolidate fragmented spans into a single, inspectable trace tree

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Prerequisites</strong>
<p style="margin: 8px 0 0 0; color: #333;">This demo builds directly on <strong>05 Demo - Building a Single Agent with the OpenAI SDK</strong>. You should be familiar with <code>Agent</code>, <code>Runner</code>, <code>McpServer</code>, and MLflow tracing from that notebook.</p>
</div>
</div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="border-left: 4px solid #f44336; background: #ffebee; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #c62828; font-size: 1.1em;">Select Compute</strong>
<p style="margin: 8px 0 0 0; color: #333;">Before starting this notebook, select the required compute environment listed below.</p>
<ul style="margin: 12px 0 0 16px; color: #333;">
<li><strong>Serverless Compute, Version 5</strong>: How to select an environment version (<a href="https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">AWS</a> | <a href="https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">Azure</a> | <a href="https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version" style="color: #1976d2; text-decoration: underline;">GCP</a>)</li>
</ul>
<p style="margin: 8px 0 0 0; color: #333;"><strong>NOTE:</strong> This notebook was <strong>developed and tested using Serverless V5</strong>. Other compute options may work but are not guaranteed to behave the same or support all features demonstrated.</p>
</div>
</div>
</div>

## A. Classroom Setup

In [0]:
%run ./Includes/Classroom-Setup-6

### A1. Import Libraries and Configure SDK

To get started with the OpenAI SDK, we have a few things that need to be setup:
1. Configure the OpenAI Agents SDK to use Databricks model routing.
1. Import MCP components needed to connect to agent tools that have already been registered to UC.
1. Initialize the Databricks `WorkspaceClient`.
1. Set the default trace processor to MLflow. MLflow integration will be discussed in section **E**. 

Additional documentation and reading:
- [`set_default_openai_client`](https://openai.github.io/openai-agents-python/ref/#agents.set_default_openai_client)
- [`set_default_openai_api`](https://openai.github.io/openai-agents-python/ref/#agents.set_default_openai_api)
- [`set_trace_processors`](https://github.com/openai/openai-agents-python/blob/main/docs/tracing.md)

In [0]:
from agents import Agent, Runner, handoff, set_default_openai_api, set_default_openai_client
from agents.tracing import set_trace_processors
from databricks_openai import AsyncDatabricksOpenAI
from databricks_openai.agents import McpServer
from databricks.sdk import WorkspaceClient

import asyncio
import nest_asyncio
import logging

nest_asyncio.apply()

set_default_openai_client(AsyncDatabricksOpenAI())
set_default_openai_api("chat_completions")
set_trace_processors([])  # Disable OpenAI trace processor; use MLflow instead
logging.getLogger("openai").setLevel(logging.WARNING)

ws = WorkspaceClient()
workspace_host = ws.config.host

print("OpenAI Agents SDK configured to use Databricks model routing.")
print(f"Workspace host: {workspace_host}")

## B. Connect to UC Tools via MCP

We can now connect our tools through Managed MCP servers. Each UC function is accessible at the endpoint of the form: `https://workspace_host/api/2.0/mcp/functions/catalog/schema`


We will build two specialized agents, each connected to a single UC function via its own MCP server:

<div style="background: #1B5162; color: white; padding: 20px 24px; border-radius: 10px; margin: 12px 0;">
  <div style="font-size: 11pt; text-transform: uppercase; letter-spacing: 1px; opacity: 0.7; margin-bottom: 6px;">Scenario</div>
  <div style="font-size: 14pt; line-height: 1.5; margin-bottom: 16px;">
    We have been tasked with building a multi-agent POC for the data team. The supervisor agent routes requests to two specialized sub-agents -- a data analyst and a business analyst -- each equipped with a UC function accessed through MCP. Select a tab below to explore each agent's role.
  </div>

  <!-- Tab Buttons -->
  <div style="display: flex; gap: 8px; margin-bottom: 0px;">
    <button onclick="showTab('supervisor')" id="btn-supervisor" style="padding: 8px 16px; border-radius: 6px 6px 0 0; border: none; cursor: pointer; font-size: 11pt; font-weight: bold; background: rgb(240,243,248); color: #1B5162;">Supervisor</button>
    <button onclick="showTab('analyst')"    id="btn-analyst"    style="padding: 8px 16px; border-radius: 6px 6px 0 0; border: none; cursor: pointer; font-size: 11pt; font-weight: bold; background: #1B5162; color: rgb(240,243,248);">Data Analyst</button>
    <button onclick="showTab('business')"   id="btn-business"   style="padding: 8px 16px; border-radius: 6px 6px 0 0; border: none; cursor: pointer; font-size: 11pt; font-weight: bold; background: #1B5162; color: rgb(240,243,248);">Business Analyst</button>
  </div>

  <!-- Tab: Supervisor -->
  <div id="tab-supervisor" style="display: block; background: rgba(255,255,255,0.08); border-radius: 0 6px 6px 6px; padding: 16px 20px; font-size: 13pt; line-height: 1.6;">
    <strong>Role:</strong> The supervisor agent is the entry point for all incoming requests. It does not hold any tools itself -- instead, it reads the intent of each request and routes it to the appropriate sub-agent.<br><br>
    <strong>Routing logic:</strong>
    <ul style="margin: 8px 0 0 0; padding-left: 20px;">
      <li>Questions about <em>pricing tiers, deal evaluation, or price classification</em> &#x2192; routed to the <strong>Data Analyst</strong></li>
      <li>Questions about <em>neighborhood summaries, average prices, listing counts, or room type mix</em> &#x2192; routed to the <strong>Business Analyst</strong></li>
    </ul>
  </div>

  <!-- Tab: Data Analyst -->
  <div id="tab-analyst" style="display: none; background: rgba(255,255,255,0.08); border-radius: 0 6px 6px 6px; padding: 16px 20px; font-size: 13pt; line-height: 1.6;">
    <strong>Role:</strong> The data analyst agent classifies Airbnb listing prices into market tiers based on SF pricing distribution. It helps users understand whether a price point is Budget, Mid-Range, Premium, or Luxury.<br><br>
    <strong>Tool:</strong> <code>classify_price_tier</code> (via MCP)<br><br>
    <strong>Returns:</strong> A JSON object containing the price, tier, percentile band, and a short interpretation.<br><br>
    <strong>Example question:</strong> <em>"Is $200/night a good deal for an Airbnb in SF?"</em>
  </div>

  <!-- Tab: Business Analyst -->
  <div id="tab-business" style="display: none; background: rgba(255,255,255,0.08); border-radius: 0 6px 6px 6px; padding: 16px 20px; font-size: 13pt; line-height: 1.6;">
    <strong>Role:</strong> The business analyst agent surfaces market insights for stakeholders. It aggregates data to answer strategic questions about neighborhoods, pricing, and listing mix.<br><br>
    <strong>Tool:</strong> <code>neighborhood_summary</code> (via MCP)<br><br>
    <strong>Returns:</strong> A table of per-neighborhood metrics including average price, total listings, and most common room type.<br><br>
    <strong>Example question:</strong> <em>"Which neighborhoods have the highest average price?"</em>
  </div>
</div>

<script>
function showTab(name) {
  ['supervisor', 'analyst', 'business'].forEach(function(t) {
    var tab = document.getElementById('tab-' + t);
    var btn = document.getElementById('btn-' + t);
    if (t === name) {
      tab.style.display = 'block';
      btn.style.background = 'rgb(240,243,248)';
      btn.style.color = '#1B5162';
    } else {
      tab.style.display = 'none';
      btn.style.background = '#1B5162';
      btn.style.color = 'rgb(240,243,248)';
    }
  });
}
</script>

### B1. Discover Available UC Functions

Before building agents, we use `DatabricksMCPClient` to verify that the UC functions are registered and discoverable. Note that in this notebook we are using two different methods for interacting with the Databricks-managed MCP server:
1. `DatabricksMCPClient`: provides a synchronous API for direct MCP tool calls (good for notebooks)
1. `McpServer`: an asynchronous context manager for integrating MCP servers with the OpenAI Agents SDK (useful for applications)

In [0]:
from databricks_mcp import DatabricksMCPClient

# Point to the MCP server for our catalog/schema
mcp_url = f"{workspace_host}/api/2.0/mcp/functions/{catalog_name}/{schema_name}"
mcp_client = DatabricksMCPClient(server_url=mcp_url, workspace_client=ws)

# Discover available tools
tools = mcp_client.list_tools()
print(f"MCP server URL: {mcp_url}")
print(f"Discovered {len(tools)} UC function tool(s):")
for t in tools:
    print(f"  - {t.name}: {t.description}")

### B2. Build MCP-Backed Agents

Each agent will have its own configuration file. Additionally, each agent will have its own `McpServer` scoped to a single UC function. This ensures each agent can only access the tool it needs:
1. `classify_price_tier` → **Data Analyst agent**
1. `neighborhood_summary` → **Business Analyst agent**

#### B2.1 Load the Agent Configuration
We will load the agent configuration file. This is a recommended practice since any updates to the agents' system prompts, name, etc. live outside this notebook. These variables will be stored in the dictionary `agents`:
- `agents["X"]['name']`: The agent's name
- `agents["X"]['llm_endpoint']`: The foundation model serving endpoint name (e.g. `databricks-gpt-X`)
- `agents["X"]['system_prompt']`: The agent's system prompt

1. Running the following cell will store config variables for the data analyst, business data analyst, and the supervisor agent that will be used later.
    - Navigate to the folder **agent_config** and inspect the agent contents for each: **business_data_analyst_agent.json**, **data_analyst_agent.json**, and **supervisor_agent.json**

In [0]:
import json
from pathlib import Path

agent_files = {
    "da": "data_analyst_agent",
    "ba": "business_data_analyst_agent",
    "sa": "supervisor_agent",
}

agents = {}
for prefix, filename in agent_files.items():
    cfg = json.loads(Path(f"./agent_config/{filename}.json").read_text())
    agents[prefix] = {
        "name": cfg["name"],
        "llm_endpoint": cfg["llm_endpoint"],
        "system_prompt": cfg["system_prompt"],
    }

#### B2.2 Data Analyst Agent

2. Run the next cell to setup the data analyst agent. 

In [0]:
da_mcp_server = McpServer.from_uc_function(
    catalog=catalog_name,
    schema=schema_name,
    function_name="classify_price_tier",
    workspace_client=ws,
    timeout=60.0,
)

data_analyst_agent = Agent(
    name=agents["da"]['name'],
    instructions=agents["da"]['system_prompt'],
    model=agents["da"]['llm_endpoint'],
    mcp_servers=[da_mcp_server],
)

#### B2.3 Business Analyst Agent

3. Run the next cell to setup the business analyst agent.

In [0]:
ba_mcp_server = McpServer.from_uc_function(
    catalog=catalog_name,
    schema=schema_name,
    function_name="neighborhood_summary",
    workspace_client=ws,
    timeout=60.0,
)

business_analyst_agent = Agent(
    name=agents["ba"]['name'],
    instructions=agents["ba"]['system_prompt'],
    model=agents["ba"]['llm_endpoint'],
    mcp_servers=[ba_mcp_server],
)

## C. Create a Supervisor Agent with Handoffs

### C1. Build the Supervisor

The supervisor agent analyzes each user request and delegates to the appropriate worker using the **handoff** mechanism from the [OpenAI Agents SDK](https://developers.openai.com/api/docs/guides/agents). When the supervisor hands off to a worker, that worker takes over the conversation and responds directly to the user.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
The <code>handoffs</code> parameter tells the supervisor which agents it can delegate to. The supervisor's instructions guide its routing decisions. The LLM reads each worker's name and instructions to decide which one is best suited for the request.
See: <a href="https://developers.openai.com/api/docs/guides/agents/orchestration" style="color: #1976d2; text-decoration: underline;">Orchestration and handoffs</a>
</p>
</div>
</div>
</div>

In [0]:
supervisor = Agent(
    name=agents["sa"]['name'],
    instructions=agents["sa"]['system_prompt'],
    model=agents["sa"]['llm_endpoint'],
    handoffs=[data_analyst_agent, business_analyst_agent],
)

print(f"Created supervisor with handoffs to: {[a.name for a in supervisor.handoffs]}")

### C2. How the Supervisor Routes Requests

The supervisor sees each worker as a **handoff tool**. When a user sends a request, the supervisor's LLM reads the request, examines the available handoffs, and decides which worker should handle it.

**Click on any component in the diagram** to learn more about it.

<style>
.ma-node {
  border-radius: 8px; padding: 12px 18px; text-align: center; font-weight: 600; font-size: 14px;
  cursor: pointer; user-select: none; transition: transform 0.12s, box-shadow 0.2s;
}
.ma-node:hover { transform: translateY(-2px); box-shadow: 0 4px 12px rgba(0,0,0,0.15); }
.ma-node.active { box-shadow: 0 0 0 3px rgba(0,0,0,0.2); transform: translateY(-2px); }
.ma-arrow { color: #999; font-size: 22px; padding: 0 8px; user-select: none; }
.ma-detail-wrap { overflow: hidden; max-height: 0; opacity: 0; transition: max-height 0.35s ease, opacity 0.28s ease, margin-top 0.28s ease; margin-top: 0; }
.ma-detail-wrap.open { max-height: 1200px; opacity: 1; margin-top: 16px; }
.ma-detail-card { background: #F9F7F4; border-radius: 10px; padding: 20px; border-top: 6px solid #ccc; font-size: 14pt; line-height: 1.7; color: #333; }
.ma-detail-card strong { color: #0b2026; }
.ma-detail-card code { background: #e8e8e8; padding: 2px 6px; border-radius: 4px; font-size: 13pt; }
.ma-example { background: #1e1e2e; color: #cdd6f4; border-radius: 8px; padding: 14px 18px; margin-top: 12px; font-family: 'Menlo','Consolas',monospace; font-size: 12pt; line-height: 1.6; }
.ma-example .q { color: #89b4fa; } .ma-example .a { color: #a6e3a1; } .ma-example .r { color: #fab387; }
</style>

<div style="max-width: 1100px; margin: 0 auto; font-family: sans-serif;">

<div style="display:flex;align-items:center;justify-content:center;gap:0;margin:18px 0;flex-wrap:wrap;">
  <div class="ma-node" style="background:#F9F7F4;border:2px solid #1B5162;color:#0b2026;" data-ma="input" onclick="maSelect('input')">
    User Query
  </div>
  <div class="ma-arrow">&#x2192;</div>
  <div class="ma-node" id="ma-supervisor" style="background:#FF5F46;color:#fff;" data-ma="supervisor" onclick="maSelect('supervisor')">
    Supervisor<br/><span style="font-weight:400;font-size:12px;">Reads instructions + handoffs</span>
  </div>
  <div class="ma-arrow">&#x2192;</div>
  <div style="display:flex;flex-direction:column;gap:6px;">
    <div class="ma-node" style="background:#2574B5;color:#fff;" data-ma="data" onclick="maSelect('data')">
      Data Analyst<br/><span style="font-weight:400;font-size:12px;">classify_price_tier via MCP</span>
    </div>
    <div class="ma-node" style="background:#02A36F;color:#fff;" data-ma="business" onclick="maSelect('business')">
      Business Analyst<br/><span style="font-weight:400;font-size:12px;">neighborhood_summary via MCP</span>
    </div>
  </div>
  <div class="ma-arrow">&#x2192;</div>
  <div class="ma-node" style="background:#F9F7F4;border:2px solid #1B5162;color:#0b2026;" data-ma="output" onclick="maSelect('output')">
    Response
  </div>
</div>

<div class="ma-detail-wrap" id="ma-detail-wrap">
  <div class="ma-detail-card" id="ma-detail-card"></div>
</div>

</div>

<script>
var MA_DATA = {
  input: {
    color: '#1B5162',
    title: 'User Query (Input)',
    html: '<strong>What happens here:</strong> The user sends a natural language question. The system does not pre-classify the query. The raw text is passed directly to the supervisor agent.<br/><br/><strong>Examples:</strong><div class="ma-example"><span class="q">"Is $200/night a good deal for an Airbnb in SF?"</span> &#x2192; routes to Data Analyst<br/><span class="q">"Which neighborhoods have the highest average price?"</span> &#x2192; routes to Business Analyst</div>'
  },
  supervisor: {
    color: '#FF5F46',
    title: 'Supervisor Agent',
    html: '<strong>Role:</strong> Analyzes every incoming request and decides which worker should handle it.<br/><br/><strong>How it decides:</strong> The supervisor has <code>handoffs=[data_analyst_agent, business_analyst_agent]</code>. The LLM sees each handoff as a callable tool named <code>transfer_to_Data_Analyst</code> and <code>transfer_to_Business_Analyst</code>. Based on the user query and the worker descriptions in its instructions, it picks the best match.<br/><br/><strong>Example routing:</strong><div class="ma-example"><span class="q">User: "Is $200/night a good deal in SF?"</span><br/><span class="r">Supervisor thinks: This is about price classification...</span><br/><span class="a">Action: transfer_to_Data_Analyst</span></div><div class="ma-example" style="margin-top:8px;"><span class="q">User: "Which neighborhoods have the highest average price?"</span><br/><span class="r">Supervisor thinks: This is a market/business question about neighborhoods...</span><br/><span class="a">Action: transfer_to_Business_Analyst</span></div>'
  },
  data: {
    color: '#2574B5',
    title: 'Data Analyst Agent',
    html: '<strong>Role:</strong> Classifies Airbnb listing prices into market tiers based on SF pricing distribution.<br/><br/><strong>Tool:</strong> <code>classify_price_tier(price: float)</code> accessed via MCP from Unity Catalog. Returns a JSON object with tier (Budget, Mid-Range, Premium, Luxury), percentile band, and interpretation.<br/><br/><strong>Example interaction:</strong><div class="ma-example"><span class="q">User: "Is $200/night a good deal in SF?"</span><br/><span class="r">Agent calls: classify_price_tier(200.0)</span><br/><span class="a">Agent: "$200/night falls in the Premium tier (50-75th percentile). This is above average for SF listings."</span></div>'
  },
  business: {
    color: '#02A36F',
    title: 'Business Analyst Agent',
    html: '<strong>Role:</strong> Surfaces market insights for stakeholders by analyzing neighborhood-level pricing and listing mix data.<br/><br/><strong>Tool:</strong> <code>neighborhood_summary(neighborhood: str)</code> accessed via MCP from Unity Catalog. Returns average price, total listings, and most common room type per neighborhood.<br/><br/><strong>Example interaction:</strong><div class="ma-example"><span class="q">User: "Which neighborhoods have the highest average price?"</span><br/><span class="r">Agent calls: neighborhood_summary()</span><br/><span class="a">Agent: "The top neighborhoods by average price are: Sea Cliff ($284.50, 89% entire homes), Pacific Heights ($263.15, 72% entire homes), and Nob Hill ($241.80, 65% entire homes)."</span></div>'
  },
  output: {
    color: '#1B5162',
    title: 'Response (Output)',
    html: '<strong>What happens here:</strong> The worker agent that received the handoff produces the final response. The response goes directly back to the user.<br/><br/><strong>Key detail:</strong> The supervisor does NOT post-process or modify the worker output. Once a handoff occurs, the worker owns the conversation. This is why <code>result.last_agent.name</code> shows the worker, not the supervisor.<br/><br/><strong>In the trace:</strong> The MLflow trace captures the full flow: supervisor LLM call &#x2192; handoff decision &#x2192; worker LLM call &#x2192; tool calls (if any) &#x2192; final output.'
  }
};
var maCurrent = null;
function maSelect(id) {
  var wrap = document.getElementById('ma-detail-wrap');
  var card = document.getElementById('ma-detail-card');
  document.querySelectorAll('.ma-node').forEach(function(n) { n.classList.toggle('active', n.dataset.ma === id); });
  if (maCurrent === id) { wrap.classList.remove('open'); document.querySelectorAll('.ma-node').forEach(function(n) { n.classList.remove('active'); }); maCurrent = null; return; }
  maCurrent = id;
  var d = MA_DATA[id];
  card.style.borderTopColor = d.color;
  card.innerHTML = '<div style="font-size:16pt;font-weight:700;margin-bottom:12px;color:' + d.color + ';">' + d.title + '</div><div>' + d.html + '</div>';
  wrap.classList.add('open');
}
</script>

## D. Test Multi-Agent Routing

### D1. Data Question

This question is about price classification. The supervisor should route it to the Data Analyst agent, which will call `classify_price_tier` via MCP. You can verify this in MLflow interface as a part of running the next cell. 

We are connecting to the Databricks managed MCP server via`McpServer`, which is an async context manager, hence we need to be mindful of using `async` and `await`. Similarly, `Runner.run()` is also async. 

In [0]:
data_query = "Is $200/night a good deal for an Airbnb in SF?"

async def test_data_query():
    async with da_mcp_server, ba_mcp_server:
        result = await Runner.run(supervisor, data_query)
        return result

result = asyncio.get_event_loop().run_until_complete(test_data_query())
print(f"User:       {data_query}")
print(f"Handled by: {result.last_agent.name}")
print(f"Response:   {result.final_output}")

### D2. Business Question

This is a market insights question. The supervisor should route it to the Business Analyst agent, which will call `neighborhood_summary` via MCP.

In [0]:
business_query = "What is the average price of the Mission neighborhood?"

async def test_business_query():
    async with da_mcp_server, ba_mcp_server:
        result = await Runner.run(supervisor, business_query)
        return result

result = asyncio.get_event_loop().run_until_complete(test_business_query())
print(f"User:       {business_query}")
print(f"Handled by: {result.last_agent.name}")
print(f"Response:   {result.final_output}")

<details style="margin: 8px 0;">
<summary style="background: linear-gradient(135deg, #1B5162, #4299E0); color: white; padding: 12px 18px; cursor: pointer; font-weight: 600; font-size: 12pt; border-radius: 8px; user-select: none;">
Why are there traces in the outputs above?
</summary>
<div style="border: 2px solid #1B5162; border-top: none; border-radius: 0 0 8px 8px; padding: 16px 20px; background: #F9F7F4; font-size: 12pt; line-height: 1.7; color: #333;">
<p style="margin-top: 10px;">Running the cell above will produce a trace showing a tool call took place. This is because <code>@mlflow.trace(span_type=SpanType.TOOL)</code> is used during <code>call_tool</code> (see the <a href="https://github.com/databricks/databricks-ai-bridge/blob/main/integrations/openai/src/databricks_openai/agents/mcp_server.py" style="color: #1976d2; text-decoration: underline;">source code</a>).</p>
</div>
</details>

### D3. Mixed Routing Test

Run multiple queries and show how the supervisor routes each one to the appropriate worker agent. Notice that in the output there are multiple traces. In the next section, we will see how to concatenate them into 1 trace and perform proper logging as a series of events so we have a more comprehensive picture of the flow from input to output via the supervisor agent.

In [0]:
import pandas as pd

test_queries = [
    ("Is $150/night a good deal in SF?", agents["da"]['name']),
    ("What is the average price in Mission?", agents["ba"]['name']),
    ("Would $350/night be considered luxury?", agents["da"]['name']),
    ("Is the average price in Mission higher or lower than in Noe Valley?", agents["ba"]['name']),
]

async def test_mixed_routing():
    async with da_mcp_server, ba_mcp_server:
        results = []
        for query, expected in test_queries:
            result = await Runner.run(supervisor, query)
            actual = result.last_agent.name
            status = "PASS" if actual == expected else "MISMATCH"
            results.append({
                "Query": query,
                "Expected Agent": expected,
                "Actual Agent": actual,
                "Status": status,
            })
        return results

results = asyncio.get_event_loop().run_until_complete(test_mixed_routing())
display(pd.DataFrame(results))

## E. Enable MLflow Tracing
Now that we have built and validated our multi-agent system, we enable autologging with MLflow to capture execution traces. This section will have fractured traces showing each handoff and tool call as a separate trace. We will see in section **F** how we can bring these together in 1 trace. For now, let's focus on the individual outputs to see how a multi-agent system appears as multiple traces. 

### E1. Enable Autologging

We enable MLflow tracing so the next agent runs produce full execution traces.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
In the MLflow Experiment UI, multi-agent traces show nested spans: the supervisor's LLM call appears at the top level, followed by the handoff event, and then the worker's LLM and tool calls as child spans. This makes it easy to debug routing decisions and tool selection.
</p>
</div>
</div>
</div>

In [0]:
import mlflow

mlflow.openai.autolog()

print("MLflow OpenAI autologging enabled.")
print(f"Traces will appear in experiment: {mlflow_experiment_path}")

### E2. Run with Tracing

Run the supervisor again with tracing enabled. The trace will capture the full handoff flow from supervisor to worker. The `transfer_to_Data_Analyst` and `transfer_to_Business_Analyst` tools do not exist in your code. The SDK generates them automatically from `handoffs=[data_analyst_agent, business_analyst_agent]` using the pattern `transfer_to_{agent.name}`.

In [0]:
async def run_with_tracing(query: str):
    async with da_mcp_server, ba_mcp_server:
        result = await Runner.run(supervisor, query)
        return result

result = asyncio.get_event_loop().run_until_complete(run_with_tracing(data_query))
print(f"User:       {data_query}")
print(f"Handled by: {result.last_agent.name}")
print(f"Response:   {result.final_output}")

## F. Distributed Tracing

Notice the output from the previous section may produce multiple separate trace outputs. MLflow supports distributed tracing to stitch spans from different calls into a single unified trace.

The key idea: wrap the entire multi-agent flow in a parent `@mlflow.trace` span. All child spans (from autologging, tool calls, and handoffs) automatically nest under it, producing one clean trace tree instead of scattered fragments.

<div style="border-left: 4px solid #1976d2; background: #e3f2fd; padding: 16px 20px; border-radius: 4px; margin: 16px 0;">
<div style="display: flex; align-items: flex-start; gap: 12px;">
<div>
<strong style="color: #0d47a1; font-size: 1.1em;">Note</strong>
<p style="margin: 8px 0 0 0; color: #333;">
For cross-service distributed tracing (e.g., an agent calling another agent over HTTP), MLflow provides <code>get_tracing_context_headers_for_http_request()</code> and <code>set_tracing_context_from_http_request_headers()</code> to propagate trace context via W3C TraceContext headers. Both services must log to the same MLflow tracking server and experiment. On Databricks, the trace destination must be set to Unity Catalog.
See: <a href="https://mlflow.org/docs/latest/genai/tracing/app-instrumentation/distributed-tracing/" style="color: #1976d2; text-decoration: underline;">MLflow Distributed Tracing docs</a>
</p>
</div>
</div>
</div>

### F1. Build Traced Agent Wrappers

To make handoffs visible in the trace, we wrap each agent's execution in its own `@mlflow.trace` span. The supervisor routes the question, then we call the selected worker inside a labeled span. This produces a trace tree like:

```
supervisor_orchestration
  +-- route_decision (supervisor LLM call)
  +-- data_analyst_execution OR business_analyst_execution
      +-- AsyncDatabricksCompletions (worker LLM + tool calls)
```

1. Create a trace wrapper for the agents defined above. Here we are setting the `span_type` to `AGENT` and defining the names of the agents as they will appear in the trace. Again, we are connecting to the Databricks managed MCP server via`McpServer`, which is an async context manager, hence we need to be mindful of using `async` and `await` as mentioned in section **D**.

In [0]:
# Traced wrapper for the Data Analyst
@mlflow.trace(span_type="AGENT", name="data_analyst_execution")
async def run_data_analyst(question: str) -> str:
    async with da_mcp_server:
        result = await Runner.run(data_analyst_agent, question)
        return result.final_output

# Traced wrapper for the Business Analyst
@mlflow.trace(span_type="AGENT", name="business_analyst_execution")
async def run_business_analyst(question: str) -> str:
    async with ba_mcp_server:
        result = await Runner.run(business_analyst_agent, question)
        return result.final_output

2. Define the router (supervisor) agent and wrap the agent with an MLflow trace as well. 

In [0]:
# Router agent (no handoffs, just classifies the request)
router_agent = Agent(
    name="Router",
    instructions=agents["sa"]['system_prompt'],
    model=agents["sa"]['llm_endpoint'],
)


@mlflow.trace(span_type="AGENT", name="supervisor_orchestration")
async def run_supervisor_traced(question: str) -> str:
    """Route to the correct agent and execute, with each step visible in the trace."""

    # Step 1: Route decision
    with mlflow.start_span(name="route_decision") as span:
        span.set_inputs({"question": question})
        route_result = await Runner.run(router_agent, question)
        route = route_result.final_output.strip().lower()
        span.set_outputs({"route": route})

    # Step 2: Execute the selected worker agent
    if "business" in route:
        return await run_business_analyst(question)
    else:
        return await run_data_analyst(question)

### F2. Run and Inspect Unified Traces

1. After running the next cell, you will a more complete picture of the flow of query handling from the supervisor agent to the subagent. the image below is an example of what you will see. The supervisor acts as the supervisor caller while the subagent, `data_analyst_execution` is the callee that handles the call to the MCP server. We also see the explicit point in the trace where the routing decision takes place `route_decision`. 

In [0]:
# Run a data question through the traced wrapper
response = await run_supervisor_traced(data_query)
print(f"Question: {data_query}")
print(f"Response: {response}")

2. As a final example, we pass a similar query to the business analyst agent. 

In [0]:
# Run a business question through the traced wrapper
response = await run_supervisor_traced(business_query)
print(f"Question: {business_query}")
print(f"Response: {response}")

## G. Conclusion
In this demo, you:

1. Connected to Unity Catalog functions (`classify_price_tier` and `neighborhood_summary`) via Managed MCP servers using both `DatabricksMCPClient` for synchronous discovery and `McpServer` for async agent integration
2. Built two specialized worker agents — a Data Analyst and a Business Analyst — each scoped to a single MCP-backed UC function
3. Created a supervisor agent that delegates user requests to the appropriate worker using the OpenAI Agents SDK `handoff` mechanism
4. Validated multi-agent routing by running data, business, and mixed queries and confirming correct handoff decisions
5. Enabled MLflow autologging and observed how a multi-agent system produces fragmented traces — one per agent invocation — without explicit span management
6. Constructed a unified trace tree by wrapping the full orchestration flow in a parent `@mlflow.trace` span, causing all child spans (routing decision, worker LLM calls, MCP tool calls) to nest under a single root


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>